# E53 --- a promessa sobre dias que nao vieram

**A tentativa.** A secao anterior construiu o readout, o fora-da-amostra e o posto do estado. Falta
a pergunta seguinte sobre o MESMO instrumento: quanto o readout pode prometer sobre dias que nao
viu?

**O que se mede.**

1. a fracao de separacoes que o readout encaixa com erro zero, contra o numero de dias, em tres
   estados --- o teto vem da contagem, e nao do numero de parametros;
2. a complexidade de Rademacher por sorteio de rotulos, contra a conta fechada raio sobre raiz dos
   dias;
3. a cota em bits: o preco da hipotese escolhida, e o prior que ja viu a metade da amostra;
4. o teto de capacidade numa sonda fresca depois de cada bloco de uso --- o controle linear, o que
   satura e o que realoca as unidades dormentes.

**Convencoes** (AGENTS.md paragrafos 7 e 9): um experimento por caderno, parametros no topo
marcados com "brinque com", algoritmo em frevolab, resultado em lab/resultados/E53_promessa.json,
figuras em .pdf e .png.

In [1]:
# <- brinque com: SERIE, JANELA, EXPONENCIAL, RESERVATORIO, GRADE_SEPARACOES, GRADE_DIAS, LAGS, DIAS_USO, BLOCOS, SIGMA, SIGMA_PRIOR, DELTA, CONFIANCA_SEPARACAO, SORTES, SEMENTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import capacidade, dados, graficos, volatilidade

SERIE = "sp500.csv"
JANELA = 250                   # a memoria que guarda bloco
EXPONENCIAL = 0.05             # a taxa da memoria exponencial
RESERVATORIO = 100             # a dimensao do reservatorio
GRADE_SEPARACOES = (2, 4, 8, 16, 32, 64, 128, 256)
GRADE_DIAS = (100, 200, 400, 800, 1600, 3200)
LAGS = 20
DIAS_USO = 6000
BLOCOS = 6
SIGMA = 0.3                    # o desvio do posterior do readout
SIGMA_PRIOR = 1.0              # o desvio do prior
DELTA = 0.05                   # a confianca declarada da cota
CONFIANCA_SEPARACAO = 0.5      # abaixo disso a fracao "desabou"
SORTES = 200
SEMENTE = 120                  # E40..E52 usam 107..119

serie = volatilidade.retornos_log(dados.carregar_serie(SERIE)).dropna()
estados = {
    "janela": capacidade.estados_janela(serie, JANELA),
    "exponencial": capacidade.estados_exponencial(serie, EXPONENCIAL),
    "reservatorio": capacidade.estados_reservatorio(serie, RESERVATORIO, semente=SEMENTE),
}
print("frevolab %s | %s: %d dias | estados: %s"
      % (frevolab.VERSAO, SERIE, serie.size,
         ", ".join("%s (%d)" % (k, v.shape[1]) for k, v in estados.items())))

frevolab 0.1.0 | sp500.csv: 6718 dias | estados: janela (250), exponencial (1), reservatorio (100)


## A contagem, e nao os parametros

A fracao de dicotomias que o readout encaixa com erro zero mede a funcao de crescimento: enquanto
os dias cabem no posto do estado, tudo cabe; depois, nada cabe.

In [2]:
separacoes, desaba = {}, {}
for nome, estado in estados.items():
    f = capacidade.fracao_de_separacoes(estado, GRADE_SEPARACOES, sortes=SORTES,
                                        semente=SEMENTE + len(nome))
    separacoes[nome] = f
    queda = [n for n, v in f.items() if v < CONFIANCA_SEPARACAO]
    desaba[nome] = queda[0] if queda else -1
    print("%-12s dimensao %3d | fracao: %s | desaba em %s"
          % (nome, estado.shape[1], {k: round(v, 2) for k, v in f.items()}, desaba[nome]))

janela       dimensao 250 | fracao: {2: 1.0, 4: 1.0, 8: 1.0, 16: 1.0, 32: 1.0, 64: 1.0, 128: 1.0, 256: 1.0} | desaba em -1
exponencial  dimensao   1 | fracao: {2: 0.52, 4: 0.12, 8: 0.01, 16: 0.0, 32: 0.0, 64: 0.0, 128: 0.0, 256: 0.0} | desaba em 4


reservatorio dimensao 100 | fracao: {2: 1.0, 4: 1.0, 8: 1.0, 16: 1.0, 32: 1.0, 64: 0.23, 128: 0.0, 256: 0.0} | desaba em 64


In [3]:
# Figura 1: a fracao de separacoes contra os dias, nos tres estados.
fig, eixo = plt.subplots(figsize=(8.6, 4.0))
for nome, f in separacoes.items():
    dias = sorted(f)
    eixo.plot(dias, [f[d] for d in dias], "o-", ms=4, label="%s (%d)" % (nome, estados[nome].shape[1]))
    eixo.axvline(estados[nome].shape[1], ls=":", lw=1.0, color="0.5")
eixo.set_xscale("log")
eixo.set_xlabel("dias da dicotomia")
eixo.set_ylabel("fracao encaixada com erro zero")
eixo.set_title("o teto vem da contagem: a queda no posto de cada estado", fontsize=10)
eixo.legend(fontsize=8)
fig.tight_layout()
graficos.salvar(fig, "E53_promessa", 1)
plt.close(fig)
print("figura E53_promessa_1 salva")

figura E53_promessa_1 salva


## O preco do dentro-fora: geometria e bits

A complexidade de Rademacher decai em raiz dos dias --- e o sorteio de rotulos e a conta fechada
sao a mesma reta. A cota em bits fecha a conta pelo outro lado: o erro fora fica abaixo do erro
dentro mais o preco, em bits, da hipotese escolhida.

In [4]:
rademacher = {}
for dias in GRADE_DIAS:
    if dias <= estados["janela"].shape[0]:
        rademacher[dias] = capacidade.rademacher_do_readout(estados["janela"].iloc[:dias],
                                                            sortes=SORTES, semente=SEMENTE)
print("%-8s %12s %12s %10s" % ("dias", "sorteio", "raio/raiz", "razao"))
for dias, r in rademacher.items():
    print("%-8d %12.4f %12.4f %10.3f"
          % (dias, r["media"], r["razao"], r["media"] / r["razao"]))

alvo = np.where(serie.shift(-1) > 0.0, 1.0, -1.0)
serie_alvo = pd.Series(alvo, index=serie.index)
cota = capacidade.cota_em_bits(estados["janela"], serie_alvo.loc[estados["janela"].index],
                               sigma=SIGMA, sigma_prior=SIGMA_PRIOR, delta=DELTA,
                               sortes=SORTES // 2, semente=SEMENTE)
for braco in ("zero", "metade"):
    b = cota[braco]
    print("%-8s dentro %.4f | fora %.4f | cota %.4f | folga %.4f | kl %.2f bits"
          % (braco, b["erro_dentro"], b["erro_fora"], b["cota"], b["folga"], b["kl_bits"]))

dias          sorteio    raio/raiz      razao
100            0.0220       0.0229      0.958
200            0.0155       0.0163      0.948
400            0.0106       0.0116      0.918
800            0.0079       0.0097      0.816
1600           0.0044       0.0069      0.642
3200           0.0037       0.0080      0.466


zero     dentro 0.3911 | fora 0.4948 | cota 0.6589 | folga 0.1641 | kl 663.71 bits
metade   dentro 0.3913 | fora 0.4955 | cota 0.7297 | folga 0.2342 | kl 1063.20 bits


In [5]:
# Figura 2: a complexidade contra os dias, e a cota em bits com a folga.
fig, eixos = plt.subplots(1, 2, figsize=(10.2, 3.9))
dias = sorted(rademacher)
eixos[0].plot(dias, [rademacher[d]["media"] for d in dias], "o-", color="#1f4e79",
              label="a complexidade por sorteio de rotulos")
eixos[0].plot(dias, [rademacher[d]["razao"] for d in dias], "--", color="#c78f2c",
              label="a conta raio sobre raiz dos dias")
eixos[0].set_xscale("log")
eixos[0].set_yscale("log")
eixos[0].set_xlabel("dias")
eixos[0].set_ylabel("complexidade")
eixos[0].set_title("o sorteio e a conta sao a mesma reta", fontsize=10)
eixos[0].legend(fontsize=7)

posicoes = np.arange(2)
largura = 0.35
eixos[1].bar(posicoes - largura / 2, [cota["zero"]["erro_fora"], cota["metade"]["erro_fora"]],
             largura, color="#1f4e79", label="o erro fora, medido")
eixos[1].bar(posicoes + largura / 2, [cota["zero"]["cota"], cota["metade"]["cota"]],
             largura, color="#c78f2c", label="a cota")
eixos[1].set_xticks(posicoes)
eixos[1].set_xticklabels(["prior no zero", "prior na metade"])
eixos[1].set_title("a cota, e o prior que abaixa o preco", fontsize=10)
eixos[1].legend(fontsize=7)
fig.tight_layout()
graficos.salvar(fig, "E53_promessa", 2)
plt.close(fig)
print("figura E53_promessa_2 salva")

figura E53_promessa_2 salva


## Com o numero de estados fixo, o teto cai com o uso

O controle e o reservatorio linear: ele nao perde nada. O que satura perde, e a realocacao das
unidades dormentes mantem o teto.

In [6]:
uso = {}
for nome, saturacao, realocacao in (("linear", False, False),
                                    ("saturado", True, False),
                                    ("realocado", True, True)):
    uso[nome] = capacidade.teto_depois_do_uso(serie.iloc[-DIAS_USO:], blocos=BLOCOS,
                                              n=RESERVATORIO, saturacao=saturacao,
                                              realocacao=realocacao, lags=LAGS,
                                              semente=SEMENTE + 7)
    t = uso[nome]
    print("%-10s teto por bloco %s | dormentes %s"
          % (nome, [round(x, 2) for x in t["teto"]], [round(x, 3) for x in t["dormentes"]]))

linear     teto por bloco [20.0, 20.0, 20.0, 20.0, 20.0, 20.0] | dormentes [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


saturado   teto por bloco [19.99, 19.74, 19.98, 19.96, 19.84, 19.33] | dormentes [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


realocado  teto por bloco [19.99, 19.74, 19.98, 19.96, 19.84, 19.33] | dormentes [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [7]:
# Figura 3: o teto medido bloco a bloco, nas tres curvas.
fig, eixo = plt.subplots(figsize=(8.6, 4.0))
for nome, t in uso.items():
    eixo.plot(t["blocos"], t["teto"], "o-", ms=4, label="%s" % nome)
eixo.set_xlabel("o bloco de uso")
eixo.set_ylabel("teto de capacidade na sonda fresca")
eixo.set_title("o teto bloco a bloco: o controle linear fica plano", fontsize=10)
eixo.legend(fontsize=8)
fig.tight_layout()
graficos.salvar(fig, "E53_promessa", 3)
plt.close(fig)
print("figura E53_promessa_3 salva")

figura E53_promessa_3 salva


## Leitura visual das figuras

**Declarada contra os .png depois da execucao** (AGENTS.md paragrafo 9).

O que as legendas do capitulo afirmam, e a leitura tem de conferir nos .png:

1. **Figura 1**: as tres curvas valendo um e desabando cada uma na sua linha pontilhada do posto.
2. **Figura 2**: no painel esquerdo as duas retas quase sobrepostas, caindo; no direito as barras do
   erro fora abaixo das barras da cota, nos dois bracos.
3. **Figura 3**: a curva do linear plana, a do saturado caindo e a do realocado acima dela.

In [8]:
# O resultado: um objeto por grandeza, em portugues, para o livro citar por comando.
ultimo_dia = sorted(rademacher)[-1]
resultado = {
    "promessa_serie": SERIE,
    "promessa_dias": int(serie.size),
    "promessa_lags": LAGS,
    "promessa_blocos": BLOCOS,
    "promessa_dimensao_reservatorio": RESERVATORIO,
    "promessa_grade_separacoes": len(GRADE_SEPARACOES),
    "promessa_desaba_janela": int(desaba["janela"]),
    "promessa_desaba_exponencial": int(desaba["exponencial"]),
    "promessa_desaba_reservatorio": int(desaba["reservatorio"]),
    "promessa_dimensao_janela": int(estados["janela"].shape[1]),
    "promessa_dimensao_exponencial": int(estados["exponencial"].shape[1]),
    "promessa_rademacher_dias": int(ultimo_dia),
    "promessa_rademacher_media": round(rademacher[ultimo_dia]["media"], 5),
    "promessa_rademacher_razao": round(rademacher[ultimo_dia]["razao"], 5),
    "promessa_rademacher_quociente": round(rademacher[ultimo_dia]["media"] / rademacher[ultimo_dia]["razao"], 3),
    "promessa_delta_pct": round(100 * DELTA, 1),
    "promessa_kl_zero_bits": round(cota["zero"]["kl_bits"], 2),
    "promessa_kl_metade_bits": round(cota["metade"]["kl_bits"], 2),
    "promessa_fora_zero": round(cota["zero"]["erro_fora"], 4),
    "promessa_fora_metade": round(cota["metade"]["erro_fora"], 4),
    "promessa_cota_zero": round(cota["zero"]["cota"], 4),
    "promessa_cota_metade": round(cota["metade"]["cota"], 4),
    "promessa_folga_zero": round(cota["zero"]["folga"], 4),
    "promessa_folga_metade": round(cota["metade"]["folga"], 4),
    "promessa_prior_abaixa_bits": round(cota["zero"]["kl_bits"] - cota["metade"]["kl_bits"], 2),
    "promessa_teto_linear_inicio": round(uso["linear"]["teto"][0], 3),
    "promessa_teto_linear_fim": round(uso["linear"]["teto"][-1], 3),
    "promessa_teto_saturado_inicio": round(uso["saturado"]["teto"][0], 3),
    "promessa_teto_saturado_fim": round(uso["saturado"]["teto"][-1], 3),
    "promessa_teto_realocado_inicio": round(uso["realocado"]["teto"][0], 3),
    "promessa_teto_realocado_fim": round(uso["realocado"]["teto"][-1], 3),
    "promessa_dormentes_media_pct": round(100 * float(np.mean(uso["saturado"]["dormentes"])), 3),
    "promessa_dormentes_fim_pct": round(100 * float(uso["saturado"]["dormentes"][-1]), 3),
    "promessa_saturado_perda_pct": round(100 * (1.0 - uso["saturado"]["teto"][-1] / max(uso["saturado"]["teto"][0], 1e-12)), 3),
    "promessa_realocado_perda_pct": round(100 * (1.0 - uso["realocado"]["teto"][-1] / max(uso["realocado"]["teto"][0], 1e-12)), 3),
}
caminho = Path("lab/resultados/E53_promessa.json")
caminho.parent.mkdir(parents=True, exist_ok=True)
caminho.write_text(json.dumps(resultado, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=1))

{
 "promessa_serie": "sp500.csv",
 "promessa_dias": 6718,
 "promessa_lags": 20,
 "promessa_blocos": 6,
 "promessa_dimensao_reservatorio": 100,
 "promessa_grade_separacoes": 8,
 "promessa_desaba_janela": -1,
 "promessa_desaba_exponencial": 4,
 "promessa_desaba_reservatorio": 64,
 "promessa_dimensao_janela": 250,
 "promessa_dimensao_exponencial": 1,
 "promessa_rademacher_dias": 3200,
 "promessa_rademacher_media": 0.00375,
 "promessa_rademacher_razao": 0.00804,
 "promessa_rademacher_quociente": 0.466,
 "promessa_delta_pct": 5.0,
 "promessa_kl_zero_bits": 663.71,
 "promessa_kl_metade_bits": 1063.2,
 "promessa_fora_zero": 0.4948,
 "promessa_fora_metade": 0.4955,
 "promessa_cota_zero": 0.6589,
 "promessa_cota_metade": 0.7297,
 "promessa_folga_zero": 0.1641,
 "promessa_folga_metade": 0.2342,
 "promessa_prior_abaixa_bits": -399.5,
 "promessa_teto_linear_inicio": 20.0,
 "promessa_teto_linear_fim": 20.0,
 "promessa_teto_saturado_inicio": 19.995,
 "promessa_teto_saturado_fim": 19.333,
 "promessa_